# Bending Fitting Function Explained
This notebook takes from a similar format as the Bending_algebra.ipynb notebook but to explain the steps behind the actual bending mode fit calculation or commonly known as the modalCorrection function.

## Setup of environment/coordinates

In [5]:
from pathlib import Path

import numpy as np
import os
#import psd_utils
from matplotlib import pyplot as plt
from scipy.interpolate import griddata
import scipy.io

#from rfcml_bending import Bending
#from rfcml_bending.force_matrix import ForceSpace

## Define the inputs for the function

In order to perform a bending mode fit, we need to read in the matrices from the forceSpace file that are results from SVD. For our function, we will define three main inputs.
* forceSpace - data set with matrices from SVD done on interaction matrix of the mirror. 
* mapOnFEGrid - resampled input surface data or unknown that we are trying to fit bending modes to.
* t - number of bending modes defined. 

The number of actuators corresponds to the number of columns in matrix U or the bending mode matrix. So we defined the actuators by the number of columns in the bending mode matrix. 

Define the maximum number of bending modes we are fitting to as the input of our function. 

In [10]:
directory = 'C:/Users/solva/OneDrive - University of Arizona/Desktop/bending_mode_priv'
file_name = 'stp_forcespace_RW_220928.mat'

for dirpath, dirnames, filenames in os.walk(directory):
    for filename in filenames:
        if filename == file_name:
            mat_file = os.path.join(dirpath, filename)
            print(dirpath)

forceSpace = scipy.io.loadmat(mat_file) # defining to a variable to be called later

maxBendingModes = 1 # defining some arbitary number of bending modes

C:/Users/solva/OneDrive - University of Arizona/Desktop/bending_mode_priv


For the purpose of demonstrating the fit calculation int his notebook, we are just going to fit a bending mode to itself.

After the parameters are set for the fitting function, we mask our input surface map to only index finite values to avoid any NaNs or Inf values. 

Our input surface map is originally a 2D matrix with the shape $N$ x $N$. With resampling done earlier in the fitting code, we interpolate to then be $N^2$ x 1. This is to fit our input surface to the dictionary of bending modes as defined by matrix $U$.



We then define a loop to go through each bending mode and perform a fitting. The loop then goes through the following steps for each bending mode. We will define the number of bending modes as $f$.

1. Populate a stiffness matrix at the $f$ x $f$ position with the inverse of the $S$ matrix from SVD calculation. 
2. The 'bendCoef' variable defined in the fit is calculated by multiplying the transpose of the $U$ matrix taking only the columns up to the number of bending modes $f$ defined. The dimensions of this calculation are [$f$ x $N^2$] x [$N^2$ x 1]. The resulting matrix is [$f$ x 1] which is an array representing how much of each mode are present in the input surface data. 
3. Now we reconstruct the mode given the input surface map information by multiplying the columns of the $U$ matrix up to the amount of bending modes that are being fitted by the 'bendCoef'. The dimensions of this calculation is [$N^2$ x $f$] x [$f$ x 1] so the resulting matrix is [$N^2$ x 1] and we can interpolate back from a 1D array to 2D array.

As for calculating the forces associated with the number of bending modes being fit to the input surface map we multiply the $V$ matrix from SVD by the inverse $S$ matrix. The 'bendCoef' array is also multiplied by the stiffness matrix up to how many bending modes are being fit to to get how much force is being is being applied to each actuator from the bending mode fit.
$F$ $=$ [$166$ x $166$] x [$166$ x $f$] x [$f$ x 1] 

The resulting matrix $F$ has dimensions of [$166$ x $f$] which represents how much of force is applied by each actuator for that given bending mode. 166 represents the amount of actuators that are available on the mirror and can be changed depending on what mirror is being used.



In [27]:
actuators = forceSpace['U'].shape[1]
sinv = np.zeros((actuators, actuators))

f = 33

mapOnFEGrid = forceSpace['U'][:, f]
maskOnFEgrid = np.isfinite(mapOnFEGrid)

print(f"Input surface shape: {mapOnFEGrid.shape}")

sinv[f, f] = 1.0/ forceSpace['S'][f, f] # Create a stiffness matrix

print(f"Stiffness matrix shape: {sinv.shape}")

bendCoef = np.dot(forceSpace['U'][:, :f].T, mapOnFEGrid)
print(f"bendCoef matrix shape: {bendCoef.shape}")

forceCoef = np.dot(sinv[:, :f], bendCoef) # Multiply each bending mode coefficient by the stiffness to get how much of each force mode
forces = np.dot(forceSpace['V'], forceCoef) # Convert from force modes to actuator forces
print(f"forces matrix shape: {forces.shape}")

#rmsForces = np.sqrt(np.dot(forces, forces) / actuators) # Calculate RMS force of all actuator forces across the mirror

forces_bal = np.dot(forceSpace['RW_af2lc'], forces)
forces_bal =  -forces_bal
#rmsForces_bal = np.sqrt(np.dot(forces_bal, forces_bal) / actuators)

zModeFit = np.dot(forceSpace['U'][:, :f], bendCoef)
fit = zModeFit - np.mean(zModeFit)




Input surface shape: (17058,)
Stiffness matrix shape: (166, 166)
bendCoef matrix shape: (33,)
forces matrix shape: (166,)
